In [24]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

## **Data Loading**

In [2]:
df = pd.read_csv(r"C:\Users\ASUS\Desktop\Sem04\Coding\sem04_Codes\UOM_sem_04\Inputs\Accident\train.csv")
df_test = pd.read_csv(r"C:\Users\ASUS\Desktop\Sem04\Coding\sem04_Codes\UOM_sem_04\Inputs\Accident\test.csv")

In [3]:
print(df.shape)
print(df_test.shape)

(517754, 14)
(172585, 13)


In [4]:
df.describe()

,id,num_lanes,curvature,speed_limit,num_reported_accidents,accident_risk
count,517754.000000,517754.000000,517754.000000,517754.000000,517754.000000,517754.000000
mean,258876.500000,2.491511,0.488719,46.112575,1.187970,0.352377
std,149462.849975,1.120434,0.272563,15.788521,0.895961,0.166417
min,0.000000,1.000000,0.000000,25.000000,0.000000,0.000000
25%,129438.250000,1.000000,0.260000,35.000000,1.000000,0.230000
50%,258876.500000,2.000000,0.510000,45.000000,1.000000,0.340000
75%,388314.750000,3.000000,0.710000,60.000000,2.000000,0.460000
max,517753.000000,4.000000,1.000000,70.000000,7.000000,1.000000


In [5]:
df.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517754 entries, 0 to 517753
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      517754 non-null  int64  
 1   road_type               517754 non-null  str    
 2   num_lanes               517754 non-null  int64  
 3   curvature               517754 non-null  float64
 4   speed_limit             517754 non-null  int64  
 5   lighting                517754 non-null  str    
 6   weather                 517754 non-null  str    
 7   road_signs_present      517754 non-null  bool   
 8   public_road             517754 non-null  bool   
 9   time_of_day             517754 non-null  str    
 10  holiday                 517754 non-null  bool   
 11  school_season           517754 non-null  bool   
 12  num_reported_accidents  517754 non-null  int64  
 13  accident_risk           517754 non-null  float64
dtypes: bool(4), float64(2), int64(4

In [7]:
print(f"Missing Values in Train = {df.isnull().sum()}")
print(f"Missing Values in Test = {df_test.isnull().sum()}")

Missing Values in Train = id                        0
road_type                 0
num_lanes                 0
curvature                 0
speed_limit               0
lighting                  0
weather                   0
road_signs_present        0
public_road               0
time_of_day               0
holiday                   0
school_season             0
num_reported_accidents    0
accident_risk             0
dtype: int64
Missing Values in Test = id                        0
road_type                 0
num_lanes                 0
curvature                 0
speed_limit               0
lighting                  0
weather                   0
road_signs_present        0
public_road               0
time_of_day               0
holiday                   0
school_season             0
num_reported_accidents    0
dtype: int64


## **Data Pre-processing**

In [8]:
# df = pd.get_dummies(df, drop_first=True)

In [9]:
# corr = df.corr()["accident_risk"].abs().sort_values(ascending=False)
# selected_features = corr[corr > 0.3].index

corr

NameError: name 'corr' is not defined

In [10]:
# --- Drop the unwanted colunms ---

drop_cols = ["road_signs_present", "id", "school_season", "num_lanes", "time_of_day"]

for col in drop_cols:
    if col in df: df = df.drop(col, axis=1)
    if col in df_test: df_test = df_test.drop(col, axis=1)

In [11]:
# --- Seperate Target from Features ---

X = df.drop("accident_risk", axis=1)
y = df["accident_risk"]

X_test = df_test.copy()

print(f"Features : {X.columns.to_list()}")
print(f"Target : accident_risk")

Features : ['road_type', 'curvature', 'speed_limit', 'lighting', 'weather', 'public_road', 'holiday', 'num_reported_accidents']
Target : accident_risk


### **Impute Missing Values**

In [12]:
# --- Impute missing values ---

num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X[num_cols] = num_imputer.fit_transform(X[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

if len(cat_cols):
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
    X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])





C:\Users\ASUS\AppData\Local\Temp\ipykernel_31676\4092527658.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns


### **Encoding**

In [13]:
# --- 1. One-hot Encoding ---

cat_cols_list = cat_cols.to_list()

if len(cat_cols_list)>0:
    X = pd.get_dummies(X, columns=cat_cols_list, drop_first=False)
    X_test = pd.get_dummies(X_test, columns=cat_cols_list, drop_first=False)

    X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)


### **Scaling**

only when KNN, SVM

In [14]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

In [15]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_sc, X_val_sc, _, _ = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

## **Model Training**

In [16]:
models = {
    "Linear Regression" : LinearRegression(),
    "KNN Regressor" : KNeighborsRegressor(),
    "Decision Tree Regressor" : DecisionTreeRegressor(),
    "Random Forest Regressor" : RandomForestRegressor(),
    "Gradient Boosting reg" : GradientBoostingRegressor(),
    "XGBoost" : XGBRegressor(),
    "CatBoost" : CatBoostRegressor()
}

results = {}

for name, model in models.items():
    if name == "KNN Regressor":
        model.fit(X_train_sc, y_train)
        y_pred = model.predict(X_val_sc)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    results[name] = r2
    print(f"{name:<30} RMSE: {rmse:.4f}  R²: {r2:.4f}")

best_model_name = max(results, key=results.get)
print(f"Best Model Name : {best_model_name}")
    

Linear Regression              RMSE: 0.0735  R²: 0.8041
KNN Regressor                  RMSE: 0.0619  R²: 0.8610
Decision Tree Regressor        RMSE: 0.0682  R²: 0.8318
Random Forest Regressor        RMSE: 0.0634  R²: 0.8546
Gradient Boosting reg          RMSE: 0.0570  R²: 0.8823
XGBoost                        RMSE: 0.0563  R²: 0.8852
Learning rate set to 0.106096
0:	learn: 0.1515881	total: 144ms	remaining: 2m 24s
1:	learn: 0.1385273	total: 174ms	remaining: 1m 26s
2:	learn: 0.1270156	total: 201ms	remaining: 1m 6s
3:	learn: 0.1169574	total: 231ms	remaining: 57.5s
4:	learn: 0.1081775	total: 260ms	remaining: 51.7s
5:	learn: 0.1005641	total: 290ms	remaining: 48s
6:	learn: 0.0937216	total: 325ms	remaining: 46.1s
7:	learn: 0.0878705	total: 369ms	remaining: 45.7s
8:	learn: 0.0828767	total: 401ms	remaining: 44.2s
9:	learn: 0.0786609	total: 430ms	remaining: 42.6s
10:	learn: 0.0750360	total: 462ms	remaining: 41.5s
11:	learn: 0.0719753	total: 491ms	remaining: 40.4s
12:	learn: 0.0693934	total: 528m

In [17]:
models = {
    "Linear Regression" : LinearRegression(),
    "KNN Regressor" : KNeighborsRegressor(n_neighbors=5),
    "Decision Tree Regressor" : DecisionTreeRegressor(max_depth=5, random_state=42),
    "Random Forest Regressor" : RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting reg" : GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost" : XGBRegressor(n_estimators=200,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            random_state=42
    ),
    "CatBoost" : CatBoostRegressor(verbose=0)
}

results = {}

for name, model in models.items():
    if name == "KNN Regressor":
        model.fit(X_train_sc, y_train)
        y_pred = model.predict(X_val_sc)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    results[name] = r2
    print(f"{name:<30} RMSE: {rmse:.4f}  R²: {r2:.4f}")

best_model_name = max(results, key=results.get)
print(f"Best Model Name : {best_model_name}")
    

Linear Regression              RMSE: 0.0735  R²: 0.8041
KNN Regressor                  RMSE: 0.0619  R²: 0.8610
Decision Tree Regressor        RMSE: 0.0641  R²: 0.8511
Random Forest Regressor        RMSE: 0.0634  R²: 0.8545
Gradient Boosting reg          RMSE: 0.0570  R²: 0.8823
XGBoost                        RMSE: 0.0565  R²: 0.8843
CatBoost                       RMSE: 0.0563  R²: 0.8854
Best Model Name : CatBoost


In [18]:
xgb_params = {
    "n_estimators" : [100, 200, 300, 400, 500],
    "max_depth" : [3,4,5,6],
    "learning_rate" : [0.01, 0.05, 0.1, 0.2],
    "subsample" : [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree" : [0.6, 0.7, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(eval_metric="rmse", random_state=42),
    param_distributions=xgb_params,
    n_iter=30,
    scoring="r2",
    cv = 5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)
print(f"Best XGB param : {xgb_search.best_params_}")
print(f"Best XGB R2 : {xgb_search.best_score_}")



Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best XGB param : {'subsample': 0.7, 'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
Best XGB R2 : 0.8863504118096731


In [26]:
# Tuning the best model

tuned_model = XGBRegressor(
    n_estimators  = 500,     # try: 100, 200, 300
    max_depth     = 6,       # try: 3, 4, 5, 6
    learning_rate = 0.05,    # try: 0.01, 0.05, 0.1, 0.2
    subsample     = 0.7,
    colsample_bytree = 0.7,
    eval_metric="logloss",
    random_state=42
)

In [27]:
cv_scores = cross_val_score(tuned_model, X, y, cv=5,
                             scoring="r2")
print(f"Cross-val scores : {cv_scores}")
print(f"Mean CV score    : {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

Cross-val scores : [0.88518198 0.88744217 0.88571592 0.88550438 0.88713484]
Mean CV score    : 0.8862 (+/- 0.0009)


### **Retrain on Full Train Dataset and Test on Test Dataset**

In [28]:
tuned_model.fit(X, y)
test_predictions = tuned_model.predict(X_test)

In [29]:
if hasattr(tuned_model, "feature_importances_"):
    importance = pd.Series(tuned_model.feature_importances_, index=X.columns)
    importance = importance.sort_values(ascending=False).head(15)
    print("\nTop 15 important features:")
    print(importance)


Top 15 important features:
lighting_night            0.354530
speed_limit               0.218738
curvature                 0.122666
lighting_dim              0.089931
weather_clear             0.077928
lighting_daylight         0.062193
num_reported_accidents    0.034874
weather_rainy             0.018037
weather_foggy             0.017925
holiday                   0.001497
public_road               0.000891
road_type_urban           0.000321
road_type_highway         0.000241
road_type_rural           0.000226
dtype: float32
